In [2]:
# 1. Install the Kaggle library in your current conda environment
!pip install -q kaggle

# 2. Create the hidden Kaggle system directory
!mkdir -p ~/.kaggle

# 3. Move the key you just uploaded via the Jupyter UI into the hidden folder
# (If your file is named 'access_token', change both names below!)
!cp kaggle.json ~/.kaggle/

# 4. Secure the key (Kaggle will throw an error if the key is publicly readable)
!chmod 600 ~/.kaggle/kaggle.json

# 5. Download the RAVDESS Video dataset
print("Downloading RAVDESS Video dataset from Kaggle...")
!kaggle datasets download -d adrivg/ravdess-emotional-speech-video -p data/raw/ravdess_videos

# 6. Unzip the downloaded file quietly (-q) and immediately delete the zip to save disk space
print("Unzipping dataset (this might take a minute)...")
!unzip -q data/raw/ravdess_videos/ravdess-emotional-speech-video.zip -d data/raw/ravdess_videos
!rm data/raw/ravdess_videos/ravdess-emotional-speech-video.zip

print("Download and extraction complete! Ready for MediaPipe.")

cp: cannot stat 'kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/adrivg/ravdess-emotional-speech-video
License(s): unknown
100%|██████████████████████████████████████| 12.4G/12.4G [08:38<00:00, 25.7MB/s]

Unzipping dataset (this might take a minute)...
Download and extraction complete! Ready for MediaPipe.


In [ ]:
# ssh -L 15052:localhost:15052 yk20q078@submit03.unibe.ch
# module load Anaconda3
# jupyter-compute 15052 --nodes 1 --ntasks 1 --time=04:00:00 --partition=gpu --gres=gpu:rtx4090:1 --mem=64G --cpus-per-task=8 --no-qos

In [3]:
# 1. Install PyTorch and TorchVision configured for GPU (CUDA 11.8)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

# 2. Install MediaPipe (Tasks API) and OpenCV (Headless for cluster compatibility)
!pip install -q mediapipe opencv-python-headless

# 3. Install the remaining data processing tools
!pip install -q numpy tqdm Pillow kaggle

print("All dependencies installed successfully! Ready for imports.")

All dependencies installed successfully! Ready for imports.


In [1]:
!nvidia-smi

Wed Apr 15 19:14:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:01:00.0 Off |                    0 |
| 30%   38C    P8             18W /  450W |       2MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import os
import cv2
import numpy as np
import urllib.request
from tqdm import tqdm

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ==========================================
# CONFIGURATION
# ==========================================
REPO_ROOT = os.getcwd()

# NEW PATH: Looking directly at your raw Kaggle download
RAW_DATA_DIR = os.path.join(REPO_ROOT, "data", "raw", "ravdess_videos", "RAVDESS dataset")
OUTPUT_DIR = os.path.join(REPO_ROOT, "data", "mesh_images")

# The specific MediaPipe AI model file needed
MODEL_DIR = os.path.join(REPO_ROOT, "models")
MODEL_PATH = os.path.join(MODEL_DIR, "face_landmarker.task")

# RAVDESS secret codes for our target emotions
EMOTION_MAP = {
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fear"
}

# ==========================================
# AUTO-DOWNLOAD THE MEDIAPIPE MODEL
# ==========================================
os.makedirs(MODEL_DIR, exist_ok=True)
if not os.path.exists(MODEL_PATH):
    print("Downloading the MediaPipe Face Landmarker AI model...")
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    urllib.request.urlretrieve(url, MODEL_PATH)
    print("Download complete!\n")

# ==========================================
# INITIALIZE MEDIAPIPE TASKS API
# ==========================================
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1
)
detector = vision.FaceLandmarker.create_from_options(options)
TESSELATION = vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION

def process_videos_to_images():
    # 1. Create the final destination folders if they don't exist
    for emo_name in EMOTION_MAP.values():
        os.makedirs(os.path.join(OUTPUT_DIR, emo_name), exist_ok=True)

    if not os.path.exists(RAW_DATA_DIR):
        print(f"❌ ERROR: Cannot find the raw videos at {RAW_DATA_DIR}")
        print("Please check your folder structure on the left side of Colab.")
        return

    # 2. Gather every single .mp4 file across all the Actor folders
    all_videos = []
    for root, dirs, files in os.walk(RAW_DATA_DIR):
        for file in files:
            if file.endswith(".mp4"):
                all_videos.append(os.path.join(root, file))

    print(f"Found {len(all_videos)} total videos. Extracting frames for our 4 target emotions...")

    # 3. Process the videos
    for video_path in tqdm(all_videos, desc="Processing RAVDESS Dataset"):
        vid_name = os.path.basename(video_path)

        # Split the filename (e.g., "01-01-03...") to find the emotion code
        parts = vid_name.split("-")
        if len(parts) < 3:
            continue

        emotion_code = parts[2]

        # If the code isn't 03, 04, 05, or 06, we ignore it!
        if emotion_code not in EMOTION_MAP:
            continue

        emo_name = EMOTION_MAP[emotion_code]
        emo_out_dir = os.path.join(OUTPUT_DIR, emo_name)

        cap = cv2.VideoCapture(video_path)
        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            # Process every 5th frame
            if frame_count % 5 == 0:
                h, w, _ = frame.shape

                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

                results = detector.detect(mp_image)

                if results.face_landmarks and len(results.face_landmarks) > 0:
                    face_landmarks = results.face_landmarks[0]
                    black_bg = np.zeros((h, w, 3), dtype=np.uint8)

                    points = []
                    for lm in face_landmarks:
                        px, py = int(lm.x * w), int(lm.y * h)
                        points.append((px, py))

                    for connection in TESSELATION:
                        start_idx, end_idx = connection.start, connection.end
                        if start_idx < len(points) and end_idx < len(points):
                            cv2.line(black_bg, points[start_idx], points[end_idx], (255, 255, 255), 1)

                    final_image = cv2.resize(black_bg, (224, 224))
                    out_filename = f"{vid_name.split('.')[0]}_frame_{frame_count}.png"
                    cv2.imwrite(os.path.join(emo_out_dir, out_filename), final_image)

            frame_count += 1
        cap.release()

if __name__ == "__main__":
    print("Starting Data Extraction...")
    process_videos_to_images()
    print("Done! Images are fully extracted and ready for PyTorch.")

Download complete!



W0000 00:00:1776271354.684436 3772175 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1776271355.097161 3772175 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776271355.124830 3772225 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 550.163.01), renderer: NVIDIA GeForce RTX 4090/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776271355.130752 3772190 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776271355.142583 3772201 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Starting Data Extraction...
Found 2880 total videos. Extracting frames for our 4 target emotions...


Processing RAVDESS Dataset: 100%|██████████| 2880/2880 [18:02<00:00,  2.66it/s] 

Done! Images are fully extracted and ready for PyTorch.


In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# ==========================================
# 1. CONFIGURATION (FIXED FOR COLAB)
# ==========================================
REPO_ROOT = os.getcwd()
DATA_DIR = os.path.join(REPO_ROOT, 'data', 'mesh_images')
SAVE_DIR = os.path.join(REPO_ROOT, 'models', 'saved_weights')
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 512
EPOCHS = 50
LEARNING_RATE = 0.0005
NUM_CLASSES = 4

# ==========================================
# 2. MODEL DEFINITION (FROZEN)
# ==========================================
def get_squeezenet_model(num_classes):
    model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT)

    # --- LES DEUX LIGNES MAGIQUES (FREEZING) ---
    for param in model.parameters():
        param.requires_grad = False
    # ------------------------------------------

    model.classifier[1] = nn.Conv2d(
        in_channels=512,
        out_channels=num_classes,
        kernel_size=(1, 1),
        stride=(1, 1)
    )

    model.num_classes = num_classes
    return model

# ==========================================
# 3. MAIN TRAINING PIPELINE
# ==========================================
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print(f"Loading images from: {DATA_DIR}")
    full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    # num_workers=2 uses multiple CPU cores to pre-fetch images in the background
    # pin_memory=True creates a direct high-speed transfer lane to the GPU

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=8,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=8,
        pin_memory=True
    )

    model = get_squeezenet_model(NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_acc = 0.0

    print(f"Starting FROZEN training for {EPOCHS} epochs...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total

        if val_acc > best_acc:
            best_acc = val_acc
            save_path = os.path.join(SAVE_DIR, "best_squeezenet_mesh_frozen_0.pth")
            torch.save(model.state_dict(), save_path)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    print(f"Complete! Max Validation Accuracy (Frozen): {best_acc:.2f}%")

Training on device: cuda
Loading images from: /storage/homefs/yk20q078/ana/data/mesh_images
Starting FROZEN training for 50 epochs...
Epoch [1/50] | Train Loss: 1.4523 | Val Acc: 29.57%
Epoch [2/50] | Train Loss: 1.3581 | Val Acc: 36.01%
Epoch [3/50] | Train Loss: 1.3160 | Val Acc: 40.44%
Epoch [4/50] | Train Loss: 1.2874 | Val Acc: 44.32%
Epoch [5/50] | Train Loss: 1.2629 | Val Acc: 44.51%
Epoch [6/50] | Train Loss: 1.2481 | Val Acc: 43.02%
Epoch [7/50] | Train Loss: 1.2358 | Val Acc: 45.51%
Epoch [8/50] | Train Loss: 1.2267 | Val Acc: 45.83%
Epoch [9/50] | Train Loss: 1.2197 | Val Acc: 45.44%
Epoch [10/50] | Train Loss: 1.2086 | Val Acc: 46.32%
Epoch [11/50] | Train Loss: 1.2066 | Val Acc: 44.74%
Epoch [12/50] | Train Loss: 1.2050 | Val Acc: 46.16%
Epoch [13/50] | Train Loss: 1.2050 | Val Acc: 46.32%
Epoch [14/50] | Train Loss: 1.1984 | Val Acc: 47.22%
Epoch [15/50] | Train Loss: 1.1945 | Val Acc: 46.79%
Epoch [16/50] | Train Loss: 1.1902 | Val Acc: 46.86%
Epoch [17/50] | Train Loss:

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# ==========================================
# 1. CONFIGURATION (FIXED FOR COLAB)
# ==========================================
REPO_ROOT = os.getcwd()
DATA_DIR = os.path.join(REPO_ROOT, 'data', 'mesh_images')
SAVE_DIR = os.path.join(REPO_ROOT, 'models', 'saved_weights')
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 512
EPOCHS = 50
LEARNING_RATE = 0.0005
NUM_CLASSES = 4

# ==========================================
# 2. MODEL DEFINITION (FULL FINE-TUNING)
# ==========================================
def get_squeezenet_model(num_classes):
    """Loads pre-trained SqueezeNet and adapts it for our specific emotion classes."""
    model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT)

    # We DO NOT freeze the layers here. The optimizer will update everything!

    # Swap out the final classification layer
    model.classifier[1] = nn.Conv2d(
        in_channels=512,
        out_channels=num_classes,
        kernel_size=(1, 1),
        stride=(1, 1)
    )
    model.num_classes = num_classes
    return model

# ==========================================
# 3. MAIN TRAINING PIPELINE
# ==========================================
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print(f"Loading images from: {DATA_DIR}")
    full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    # num_workers=2 uses multiple CPU cores to pre-fetch images in the background
    # pin_memory=True creates a direct high-speed transfer lane to the GPU

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=8,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=8,
        pin_memory=True
    )

    model = get_squeezenet_model(NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_acc = 0.0

    print(f"Starting FULL training for {EPOCHS} epochs...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total

        if val_acc > best_acc:
            best_acc = val_acc
            save_path = os.path.join(SAVE_DIR, "best_squeezenet_mesh_full.pth")
            torch.save(model.state_dict(), save_path)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    print(f"Complete! Max Validation Accuracy (Full): {best_acc:.2f}%")

Training on device: cuda
Loading images from: /storage/homefs/yk20q078/ana/data/mesh_images
Starting FULL training for 50 epochs...
Epoch [1/50] | Train Loss: 1.4193 | Val Acc: 25.99%
Epoch [2/50] | Train Loss: 1.3771 | Val Acc: 32.29%
Epoch [3/50] | Train Loss: 1.3473 | Val Acc: 34.32%
Epoch [4/50] | Train Loss: 1.2920 | Val Acc: 38.43%
Epoch [5/50] | Train Loss: 1.3571 | Val Acc: 32.20%
Epoch [6/50] | Train Loss: 1.3486 | Val Acc: 33.30%
Epoch [7/50] | Train Loss: 1.3277 | Val Acc: 33.19%
Epoch [8/50] | Train Loss: 1.3199 | Val Acc: 33.97%
Epoch [9/50] | Train Loss: 1.3065 | Val Acc: 34.94%
Epoch [10/50] | Train Loss: 1.2715 | Val Acc: 44.40%
Epoch [11/50] | Train Loss: 1.1779 | Val Acc: 44.19%
Epoch [12/50] | Train Loss: 1.1336 | Val Acc: 49.17%
Epoch [13/50] | Train Loss: 1.0839 | Val Acc: 51.65%
Epoch [14/50] | Train Loss: 1.0508 | Val Acc: 48.58%
Epoch [15/50] | Train Loss: 1.0369 | Val Acc: 55.15%
Epoch [16/50] | Train Loss: 0.9843 | Val Acc: 59.56%
Epoch [17/50] | Train Loss: 0

In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, Dataset
from PIL import Image

# ==========================================
# 1. CONFIGURATION
# ==========================================
REPO_ROOT = os.getcwd()
DATA_DIR = os.path.join(REPO_ROOT, 'data', 'mesh_images')
SAVE_DIR = os.path.join(REPO_ROOT, 'models', 'saved_weights')
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 512
EPOCHS = 50
LEARNING_RATE = 0.0005
NUM_CLASSES = 4

# ==========================================
# 2. RAW BYTE IN-MEMORY DATASET (CRASH-PROOF)
# ==========================================
class FastRamDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        temp_dataset = datasets.ImageFolder(root=root_dir)

        self.images = []
        self.labels = []

        # We resize and convert to raw bytes ONCE to save memory
        prep_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.PILToTensor() # Saves as uint8 (1 byte per pixel) instead of float32
        ])

        print("Optimizing RAM: Loading raw pixels (This takes ~1-2 minutes)...")
        for path, label in temp_dataset.samples:
            img = Image.open(path).convert('RGB')
            img_tensor = prep_transform(img)

            self.images.append(img_tensor)
            self.labels.append(label)
            img.close()

        print(f"Success! {len(self.images)} images loaded as raw bytes.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Fetch the raw bytes
        img = self.images[idx]
        label = self.labels[idx]

        # Convert raw bytes (0-255) to AI decimals (0.0-1.0) on the fly
        img = img.float() / 255.0

        # Apply data augmentation (like flips) dynamically
        if self.transform:
            img = self.transform(img)

        return img, label

# ==========================================
# 3. MODEL DEFINITION
# ==========================================
def get_squeezenet_model(num_classes):
    model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT)

    # Freeze the base layers
    for param in model.parameters():
        param.requires_grad = False

    model.classifier[1] = nn.Conv2d(
        in_channels=512,
        out_channels=num_classes,
        kernel_size=(1, 1),
        stride=(1, 1)
    )
    model.num_classes = num_classes
    return model

# ==========================================
# 4. MAIN TRAINING PIPELINE
# ==========================================
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    # Notice: Resize and ToTensor are removed here because the FastRamDataset handles it!
    runtime_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print(f"Finding images in: {DATA_DIR}")

    full_dataset = FastRamDataset(root_dir=DATA_DIR, transform=runtime_transform)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    # num_workers=0 because the data is already in RAM. No background fetching needed!
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = get_squeezenet_model(NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_acc = 0.0

    print(f"Starting FROZEN training for {EPOCHS} epochs...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total

        if val_acc > best_acc:
            best_acc = val_acc
            save_path = os.path.join(SAVE_DIR, "best_squeezenet_mesh_frozen.pth")
            torch.save(model.state_dict(), save_path)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    print(f"Complete! Max Validation Accuracy: {best_acc:.2f}%")

Training on device: cuda
Finding images in: /storage/homefs/yk20q078/ana/data/mesh_images
Optimizing RAM: Loading raw pixels (This takes ~1-2 minutes)...
Success! 34526 images loaded as raw bytes.
Starting FROZEN training for 50 epochs...
Epoch [1/50] | Train Loss: 1.3920 | Val Acc: 32.49%
Epoch [2/50] | Train Loss: 1.3390 | Val Acc: 40.20%
Epoch [3/50] | Train Loss: 1.2882 | Val Acc: 42.92%
Epoch [4/50] | Train Loss: 1.2592 | Val Acc: 44.14%
Epoch [5/50] | Train Loss: 1.2454 | Val Acc: 43.18%
Epoch [6/50] | Train Loss: 1.2360 | Val Acc: 45.67%
Epoch [7/50] | Train Loss: 1.2242 | Val Acc: 45.48%
Epoch [8/50] | Train Loss: 1.2170 | Val Acc: 46.34%
Epoch [9/50] | Train Loss: 1.2116 | Val Acc: 46.41%
Epoch [10/50] | Train Loss: 1.2051 | Val Acc: 46.15%
Epoch [11/50] | Train Loss: 1.2033 | Val Acc: 46.76%
Epoch [12/50] | Train Loss: 1.1959 | Val Acc: 46.41%
Epoch [13/50] | Train Loss: 1.1969 | Val Acc: 47.55%
Epoch [14/50] | Train Loss: 1.1956 | Val Acc: 47.26%
Epoch [15/50] | Train Loss: 

In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, Dataset
from PIL import Image

# ==========================================
# 1. CONFIGURATION
# ==========================================
REPO_ROOT = os.getcwd()
DATA_DIR = os.path.join(REPO_ROOT, 'data', 'mesh_images')
SAVE_DIR = os.path.join(REPO_ROOT, 'models', 'saved_weights')
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 512
EPOCHS = 50
LEARNING_RATE = 0.0005
NUM_CLASSES = 4

# ==========================================
# 2. RAW BYTE IN-MEMORY DATASET (CRASH-PROOF)
# ==========================================
class FastRamDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        temp_dataset = datasets.ImageFolder(root=root_dir)

        self.images = []
        self.labels = []

        # We resize and convert to raw bytes ONCE to save memory
        prep_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.PILToTensor() # Saves as uint8 (1 byte per pixel) instead of float32
        ])

        print("Optimizing RAM: Loading raw pixels (This takes ~1-2 minutes)...")
        for path, label in temp_dataset.samples:
            img = Image.open(path).convert('RGB')
            img_tensor = prep_transform(img)

            self.images.append(img_tensor)
            self.labels.append(label)
            img.close()

        print(f"Success! {len(self.images)} images loaded as raw bytes.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Fetch the raw bytes
        img = self.images[idx]
        label = self.labels[idx]

        # Convert raw bytes (0-255) to AI decimals (0.0-1.0) on the fly
        img = img.float() / 255.0

        # Apply data augmentation (like flips) dynamically
        if self.transform:
            img = self.transform(img)

        return img, label

# ==========================================
# 3. MODEL DEFINITION
# ==========================================
def get_squeezenet_model(num_classes):
    """Loads pre-trained SqueezeNet and adapts it for our specific emotion classes."""
    model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT)

    # We DO NOT freeze the layers here. The optimizer will update everything!

    # Swap out the final classification layer
    model.classifier[1] = nn.Conv2d(
        in_channels=512,
        out_channels=num_classes,
        kernel_size=(1, 1),
        stride=(1, 1)
    )
    model.num_classes = num_classes
    return model

# ==========================================
# 4. MAIN TRAINING PIPELINE
# ==========================================
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    # Notice: Resize and ToTensor are removed here because the FastRamDataset handles it!
    runtime_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print(f"Finding images in: {DATA_DIR}")

    full_dataset = FastRamDataset(root_dir=DATA_DIR, transform=runtime_transform)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    # num_workers=0 because the data is already in RAM. No background fetching needed!
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = get_squeezenet_model(NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_acc = 0.0

    print(f"Starting FROZEN training for {EPOCHS} epochs...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total

        if val_acc > best_acc:
            best_acc = val_acc
            save_path = os.path.join(SAVE_DIR, "best_squeezenet_mesh_frozen_2.pth")
            torch.save(model.state_dict(), save_path)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    print(f"Complete! Max Validation Accuracy: {best_acc:.2f}%")

Training on device: cuda
Finding images in: /storage/homefs/yk20q078/ana/data/mesh_images
Optimizing RAM: Loading raw pixels (This takes ~1-2 minutes)...
Success! 34526 images loaded as raw bytes.
Starting FROZEN training for 50 epochs...
Epoch [1/50] | Train Loss: 1.3948 | Val Acc: 32.90%
Epoch [2/50] | Train Loss: 1.3578 | Val Acc: 33.77%
Epoch [3/50] | Train Loss: 1.3408 | Val Acc: 34.78%
Epoch [4/50] | Train Loss: 1.3279 | Val Acc: 37.13%
Epoch [5/50] | Train Loss: 1.3154 | Val Acc: 35.30%
Epoch [6/50] | Train Loss: 1.3014 | Val Acc: 40.05%
Epoch [7/50] | Train Loss: 1.2741 | Val Acc: 42.12%
Epoch [8/50] | Train Loss: 1.2049 | Val Acc: 34.87%
Epoch [9/50] | Train Loss: 1.1945 | Val Acc: 48.93%
Epoch [10/50] | Train Loss: 1.0765 | Val Acc: 54.63%
Epoch [11/50] | Train Loss: 1.0005 | Val Acc: 57.65%
Epoch [12/50] | Train Loss: 0.9504 | Val Acc: 61.05%
Epoch [13/50] | Train Loss: 0.9305 | Val Acc: 61.83%
Epoch [14/50] | Train Loss: 0.8775 | Val Acc: 64.28%
Epoch [15/50] | Train Loss: 

In [7]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# ==========================================
# 1. CONFIGURATION (FIXED FOR COLAB)
# ==========================================
REPO_ROOT = os.getcwd()
DATA_DIR = os.path.join(REPO_ROOT, 'data', 'mesh_images')
SAVE_DIR = os.path.join(REPO_ROOT, 'models', 'saved_weights')
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 512
EPOCHS = 150
LEARNING_RATE = 0.0005
NUM_CLASSES = 4

# ==========================================
# 2. MODEL DEFINITION (FULL FINE-TUNING)
# ==========================================
def get_squeezenet_model(num_classes):
    """Loads pre-trained SqueezeNet and adapts it for our specific emotion classes."""
    model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT)

    # We DO NOT freeze the layers here. The optimizer will update everything!

    # Swap out the final classification layer
    model.classifier[1] = nn.Conv2d(
        in_channels=512,
        out_channels=num_classes,
        kernel_size=(1, 1),
        stride=(1, 1)
    )
    model.num_classes = num_classes
    return model

# ==========================================
# 3. MAIN TRAINING PIPELINE
# ==========================================
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print(f"Loading images from: {DATA_DIR}")
    full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    # num_workers=2 uses multiple CPU cores to pre-fetch images in the background
    # pin_memory=True creates a direct high-speed transfer lane to the GPU

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=8,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=8,
        pin_memory=True
    )

    model = get_squeezenet_model(NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_acc = 0.0

    print(f"Starting FULL training for {EPOCHS} epochs...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total

        if val_acc > best_acc:
            best_acc = val_acc
            save_path = os.path.join(SAVE_DIR, "best_squeezenet_mesh_full_2.pth")
            torch.save(model.state_dict(), save_path)

        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    print(f"Complete! Max Validation Accuracy (Full): {best_acc:.2f}%")

Training on device: cuda
Loading images from: /storage/homefs/yk20q078/ana/data/mesh_images
Starting FULL training for 150 epochs...
Epoch [1/150] | Train Loss: 1.3830 | Val Acc: 33.97%
Epoch [2/150] | Train Loss: 1.3324 | Val Acc: 38.69%
Epoch [3/150] | Train Loss: 1.2380 | Val Acc: 44.32%
Epoch [4/150] | Train Loss: 1.1441 | Val Acc: 49.97%
Epoch [5/150] | Train Loss: 1.0748 | Val Acc: 53.84%
Epoch [6/150] | Train Loss: 1.0218 | Val Acc: 58.25%
Epoch [7/150] | Train Loss: 0.9746 | Val Acc: 57.17%
Epoch [8/150] | Train Loss: 0.9391 | Val Acc: 60.77%
Epoch [9/150] | Train Loss: 0.8947 | Val Acc: 63.15%
Epoch [10/150] | Train Loss: 0.8631 | Val Acc: 65.60%
Epoch [11/150] | Train Loss: 0.8388 | Val Acc: 64.87%
Epoch [12/150] | Train Loss: 0.8222 | Val Acc: 66.17%
Epoch [13/150] | Train Loss: 0.7943 | Val Acc: 67.67%
Epoch [14/150] | Train Loss: 0.7823 | Val Acc: 67.62%
Epoch [15/150] | Train Loss: 0.7655 | Val Acc: 66.12%
Epoch [16/150] | Train Loss: 0.7553 | Val Acc: 68.17%
Epoch [17/15

In [9]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# ==========================================
# 1. CONFIGURATION
# ==========================================
REPO_ROOT = os.getcwd()
DATA_DIR = os.path.join(REPO_ROOT, 'data', 'mesh_images')
SAVE_DIR = os.path.join(REPO_ROOT, 'models', 'saved_weights')
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 512
EPOCHS = 250
LEARNING_RATE = 0.0005
NUM_CLASSES = 4

# ==========================================
# 2. MODEL DEFINITION (FULL FINE-TUNING)
# ==========================================
def get_squeezenet_model(num_classes):
    """Loads pre-trained SqueezeNet and adapts it for our specific emotion classes."""
    model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT)

    # We DO NOT freeze the layers here. The optimizer will update everything!

    # Swap out the final classification layer
    model.classifier[1] = nn.Conv2d(
        in_channels=512,
        out_channels=num_classes,
        kernel_size=(1, 1),
        stride=(1, 1)
    )
    model.num_classes = num_classes
    return model

# ==========================================
# 3. MAIN TRAINING PIPELINE
# ==========================================
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print(f"Loading images from: {DATA_DIR}")
    full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=8,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=8,
        pin_memory=True
    )

    model = get_squeezenet_model(NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- NEW: THE LEARNING RATE SCHEDULER ---
    # mode='max': We want the scheduler to watch the validation ACCURACY (higher is better).
    # factor=0.5: When it plateaus, cut the learning rate in half.
    # patience=5: Wait for 5 epochs of no improvement before cutting the rate.
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max', 
        factor=0.5, 
        patience=5
    )
    # ----------------------------------------

    best_acc = 0.0

    print(f"Starting FULL training for {EPOCHS} epochs...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total
        
        # --- NEW: STEP THE SCHEDULER ---
        # We feed the validation accuracy to the scheduler so it knows how we are doing
        scheduler.step(val_acc)
        
        # Grab the current learning rate to print it
        current_lr = optimizer.param_groups[0]['lr']
        # -------------------------------

        if val_acc > best_acc:
            best_acc = val_acc
            save_path = os.path.join(SAVE_DIR, "best_squeezenet_mesh_full_3.pth")
            torch.save(model.state_dict(), save_path)

        # Added the current Learning Rate (LR) to the print statement
        print(f"Epoch [{epoch+1}/{EPOCHS}] | LR: {current_lr:.6f} | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    print(f"Complete! Max Validation Accuracy (Full): {best_acc:.2f}%")

Training on device: cuda
Loading images from: /storage/homefs/yk20q078/ana/data/mesh_images
Starting FULL training for 250 epochs...
Epoch [1/250] | LR: 0.000500 | Train Loss: 1.3843 | Val Acc: 36.49%
Epoch [2/250] | LR: 0.000500 | Train Loss: 1.3373 | Val Acc: 36.66%
Epoch [3/250] | LR: 0.000500 | Train Loss: 1.3173 | Val Acc: 37.49%
Epoch [4/250] | LR: 0.000500 | Train Loss: 1.2898 | Val Acc: 41.38%
Epoch [5/250] | LR: 0.000500 | Train Loss: 1.1828 | Val Acc: 49.87%
Epoch [6/250] | LR: 0.000500 | Train Loss: 1.0862 | Val Acc: 51.67%
Epoch [7/250] | LR: 0.000500 | Train Loss: 1.0354 | Val Acc: 52.85%
Epoch [8/250] | LR: 0.000500 | Train Loss: 1.0024 | Val Acc: 55.31%
Epoch [9/250] | LR: 0.000500 | Train Loss: 0.9446 | Val Acc: 60.05%
Epoch [10/250] | LR: 0.000500 | Train Loss: 0.9217 | Val Acc: 61.74%
Epoch [11/250] | LR: 0.000500 | Train Loss: 0.8674 | Val Acc: 63.37%
Epoch [12/250] | LR: 0.000500 | Train Loss: 0.8281 | Val Acc: 64.29%
Epoch [13/250] | LR: 0.000500 | Train Loss: 0.80